# Udeco 多尺度物理分量分解实验

本 Notebook 用于检查单路 Udeco 信号能否近似拆分为

$$
U(t)=B_{\mathrm{slow}}(t)+P(t)+J(t)+n(t)
$$

- $J(t)$：接触或脱离引起的快速、有限时长跃迁；
- $P(t)$：压迫增强、保持和松弛造成的中尺度变化；
- $B_{\mathrm{slow}}(t)$：汗液、温度和长期界面变化产生的超慢漂移；
- $n(t)$：上述成分之外的剩余噪声。

主线固定为 **Hampel → Multi-scale Ramp/Haar → Robust LOWESS → Conditional Bslow**。
Udeco2 与 Udeco3 使用同一套函数，但分别估计局部噪声、事件和基线。


In [ ]:
# ============================================================
# 用户配置：正常情况下只需修改 SAMPLE_NAME，然后从头运行 Notebook
# 所有时间参数均使用秒；算法内部会根据实际 fs 自动换算为采样点。
# ============================================================

SAMPLE_NAME = "ts_write_1"

# 数据根目录。增减无算法意义；仅在数据目录移动时修改。
# 推荐：保持为仓库共享 subject_data，或通过 UDECO_DATA_ROOT 环境变量覆盖。
DATA_ROOT = r"D:\MHLcode\subject_data"

# Pydisplay 协议采样率（Hz）。当前仓库数据固定为 100 Hz；通常不要修改。
# 增大/减小会按比例改变所有秒制窗口对应的采样点数；推荐仅使用真实采样率。
FS_HZ = 100

# =========================
# 预处理
# =========================
# Hampel 局部窗口（秒）。增大：局部中位数更稳但更可能把短边沿视为异常；
# 减小：更保护快速变化但对宽一点的尖峰不敏感。推荐 0.03–0.11 s。
HAMPEL_WINDOW_SEC = 0.05

# Hampel 异常阈值（局部稳健标准差倍数）。增大：只删除极端尖峰；
# 减小：删除更多点但可能误伤真实变化。推荐 3–6，第一版取 4.5。
HAMPEL_N_SIGMA = 4.5

# =========================
# J：快速接触/脱离跃迁
# =========================
# 候选过渡时长（秒）。较大尺度能发现较慢接触，较小尺度更敏感于快速边沿；
# 范围过宽会增加重复候选。100 Hz 数据推荐覆盖 0.05–0.40 s。
EDGE_SCALES_SEC = [0.05, 0.1, 0.2, 0.4]

# 过渡前稳定平台窗口（秒）。增大：平台中位数更稳但容易混入邻近事件；
# 减小：定位更局部但 MAD 波动更大。推荐 0.15–0.80 s。
EDGE_PRE_WINDOW_SEC = 0.30

# 过渡后平台/持续性窗口（秒）。增大：更严格确认新平台保持；
# 减小：允许更短平台但误检风险上升。推荐 0.20–1.00 s。
EDGE_POST_WINDOW_SEC = 0.50

# 前后平台差的局部 MAD 阈值。增大：减少误检但漏掉弱跃迁；
# 减小：检出更灵敏但噪声候选更多。推荐 4–8。
EDGE_Z_THRESHOLD = 5.0

# 接受事件的最短实际跃迁时长（秒）。增大：排除更快变化；
# 减小：更接近单点阶跃。100 Hz 下推荐 0.02–0.10 s，取 0.03 s。
EDGE_MIN_DURATION_SEC = 0.03

# 有向 Spearman 单调性阈值，范围 0–1。增大：只接受更平滑的单向变化；
# 减小：容忍抖动但可能接收无方向波动。推荐 0.50–0.85。
EDGE_MONOTONICITY_THRESHOLD = 0.65

# 前/后平台斜率相对于过渡区斜率的最大比例。增大：更容忍平台本身缓变，
# 但可能把长压迫坡切成 J；减小：更强调平平台但会漏掉漂移背景上的边沿。推荐 0.20–0.50。
EDGE_PLATFORM_SLOPE_RATIO = 0.35

# J 事件两侧额外屏蔽时长（秒）。增大：Bm 更不受边沿残差影响；
# 减小：保留更多有效趋势点。推荐 0.05–0.50 s。
EDGE_MASK_MARGIN_SEC = 0.15

# =========================
# Bm：中尺度趋势
# =========================
# 中尺度趋势算法。第一版固定为 robust_loess；保留该参数便于未来替换实现。
MEDIUM_METHOD = "robust_loess"

# LOWESS 等效局部窗口（秒）。增大：Bm 更平滑但可能削弱压迫缓坡；
# 减小：更贴近原信号但残留更多高频噪声。推荐 0.5–5.0 s。
MEDIUM_WINDOW_SEC = 2.0

# LOWESS 残差稳健重加权次数。增大：更抗离群点但更慢；
# 减小：计算更快但更易受异常拖动。推荐 1–4 次。
MEDIUM_ROBUST_ITERS = 2

# =========================
# Bslow：超慢基线
# =========================
# 条件 EWMA 的名义时间常数（秒）。增大：基线更慢、更不吸收压迫；
# 减小：更快跟踪长期漂移但可能侵蚀 P。推荐 30–300 s。
BSLOW_TAU_SEC = 120.0

# 名义 EWMA 更新倍率，实际 alpha 仍按 fs 和 tau 换算。增大：稳定段跟踪更快；
# 减小：更新更保守。推荐 0.1–1.0，且必须位于 (0, 1]。
BSLOW_UPDATE_RATE = 1.0

# 允许更新时的最大绝对 edge score。增大：边沿附近也可能更新；
# 减小：冻结范围更保守。推荐 1.5–4.0，通常低于事件检测阈值。
BSLOW_STABLE_EDGE_THRESHOLD = 2.5

# 允许更新时的最大 |dBm/dt|（Udeco/s）。增大：压迫缓坡中也可能更新；
# 减小：只在更平坦区更新。推荐 5e-5–1e-3，需结合调试图调整。
BSLOW_STABLE_SLOPE_THRESHOLD = 2e-4

# 允许更新时 Bm 相对当前 Bslow 的最大偏移（Udeco）。增大：高平台可能被吸收；
# 减小：基线更易长期冻结。推荐 1e-4–2e-3。
BSLOW_STABLE_OFFSET_THRESHOLD = 5e-4

# =========================
# 可视化
# =========================
# 局部观察起止时间（秒）。None 表示使用记录边界；增减只影响绘图，不影响算法。
PLOT_START_SEC = None
PLOT_END_SEC = None

# False 显示真实 Udeco 数值；True 仅对绘图副本做稳健标准化，便于比较形态。
# 该开关绝不会修改 result2/result3 中的实际分解结果。
NORMALIZE_COMPONENTS_FOR_PLOT = False


## 环境与数据读取

数据读取完全复用 `udeco_analysis.load_udeco_sample`。它负责序号修复、缺失点插值、
读取共享 `ck.mat` 标定并计算 Udeco2/Udeco3；本 Notebook 不重复实现这些仓库契约。


In [ ]:
from __future__ import annotations

import math
import os
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import statsmodels
from IPython.display import display
from scipy.stats import spearmanr
from statsmodels.nonparametric.smoothers_lowess import lowess

from udeco_analysis import load_udeco_sample

print(
    f"Python {platform.python_version()} | NumPy {np.__version__} | "
    f"SciPy {scipy.__version__} | statsmodels {statsmodels.__version__}"
)


In [ ]:
# 环境变量优先，便于换机器；第一单元的 DATA_ROOT 仍是可见且统一的默认值。
data_root = Path(os.environ.get("UDECO_DATA_ROOT", DATA_ROOT)).expanduser().resolve()
if not data_root.is_dir():
    raise NotADirectoryError(f"数据根目录不存在：{data_root}")

# SAMPLE_NAME 不含 `_sensor.csv`。只接受唯一匹配，避免静默读错受试者或动作批次。
sample_matches = sorted(data_root.glob(f"*/{SAMPLE_NAME}_sensor.csv"))
if not sample_matches:
    raise FileNotFoundError(
        f"没有找到样本 {SAMPLE_NAME!r}；期望路径形如 "
        f"{data_root}/<subject>/{SAMPLE_NAME}_sensor.csv"
    )
if len(sample_matches) > 1:
    raise RuntimeError(f"样本名不唯一：{[str(path) for path in sample_matches]}")

sensor_file = sample_matches[0]
sample = load_udeco_sample(sensor_file, data_root=data_root, fs=FS_HZ)

# 后续算法只依赖这四个变量，因此切换样本不需要改动任何函数调用。
time = sample.time_s.copy()
udeco2 = sample.udeco2.copy()
udeco3 = sample.udeco3.copy()
fs = sample.fs

if not (len(time) == len(udeco2) == len(udeco3)):
    raise RuntimeError("time、Udeco2、Udeco3 长度不一致。")
if len(time) == 0 or not (np.isfinite(udeco2).all() and np.isfinite(udeco3).all()):
    raise RuntimeError("Udeco 数据为空或包含无法恢复的非有限值。")


In [ ]:
# 只输出紧凑摘要，避免在 Notebook 中打印整段原始数组。
load_summary = pd.Series(
    {
        "sample_name": sample.sample_name,
        "sensor_file": str(sensor_file),
        "fs_hz": fs,
        "n_samples": len(time),
        "duration_sec": time[-1] if len(time) else 0.0,
        "Udeco2_min": float(np.min(udeco2)),
        "Udeco2_max": float(np.max(udeco2)),
        "Udeco3_min": float(np.min(udeco3)),
        "Udeco3_max": float(np.max(udeco3)),
        "sequence_interpolated": int(sample.qc["InterpFlag"].sum()),
    },
    name="value",
)
display(load_summary.to_frame())


## 后续分解路径

1. Hampel 仅去除孤立尖峰，保留 `raw` 与 `clean`；
2. 多尺度平台—过渡—平台检测重建累计跃迁分量 $J(t)$；
3. 在减去 J 并弱化事件区域后，用 Robust LOWESS 估计 $B_m(t)$；
4. 仅在真正稳定区更新 $B_{\mathrm{slow}}(t)$，最后得到 $P=B_m-B_{\mathrm{slow}}$。

后续阶段会逐层加入函数、关键合成检查和诊断图。


## 1. Hampel 轻度去除孤立尖峰

这一层只处理明显的单采样点异常，不对整段信号做低通。局部中位数提供参考，
局部 MAD 提供与通道幅度自适应的噪声尺度；连续异常点不会在第一版中被强行替换，
从而尽量避免把真实的多采样点接触边沿削平。

In [ ]:
def _require_finite_signal(signal, *, name="signal"):
    """将输入统一为一维浮点数组，并尽早暴露空数组或 NaN 等输入问题。"""
    values = np.asarray(signal, dtype=float).reshape(-1)
    if values.size == 0:
        raise ValueError(f"{name} 不能为空。")
    if not np.isfinite(values).all():
        raise ValueError(f"{name} 必须全部为有限数值。")
    return values


def _seconds_to_samples(seconds, fs, *, minimum=1, odd=False):
    """把秒换算为采样点；需要中值窗口时可强制为奇数，保证窗口左右对称。"""
    if not np.isfinite(seconds) or seconds <= 0 or int(fs) <= 0:
        raise ValueError("时间参数和 fs 必须为正数。")
    samples = max(int(minimum), int(round(float(seconds) * int(fs))))
    if odd and samples % 2 == 0:
        samples += 1
    return samples


def _robust_sigma(values, *, epsilon=1e-12):
    """使用 1.4826×MAD 估计稳健标准差，并提供数值下限。"""
    values = np.asarray(values, dtype=float)
    center = float(np.median(values))
    return max(float(epsilon), 1.4826 * float(np.median(np.abs(values - center))))


def hampel_filter(signal, *, fs, window_sec, n_sigma):
    """仅替换 Hampel 判定为异常且前后均非异常的单采样点尖峰。

    参数增大/减小的效果由首个参数单元统一说明。返回新数组，不修改输入信号。
    """
    values = _require_finite_signal(signal)
    if not np.isfinite(n_sigma) or n_sigma <= 0:
        raise ValueError("n_sigma 必须为正数。")

    window = _seconds_to_samples(window_sec, fs, minimum=3, odd=True)
    series = pd.Series(values)
    minimum_points = max(2, window // 2 + 1)
    local_median = series.rolling(window, center=True, min_periods=minimum_points).median()
    absolute_deviation = (series - local_median).abs()
    local_mad = absolute_deviation.rolling(
        window, center=True, min_periods=minimum_points
    ).median()
    local_sigma = 1.4826 * local_mad.to_numpy(dtype=float)

    # 当局部窗口完全平坦、MAD 为 0 时，使用全局短时残差尺度，避免除零。
    median_values = local_median.to_numpy(dtype=float)
    global_sigma = _robust_sigma(values - np.where(np.isfinite(median_values), median_values, values))
    safe_sigma = np.where(np.isfinite(local_sigma) & (local_sigma > 0), local_sigma, global_sigma)
    candidates = np.abs(values - median_values) > float(n_sigma) * safe_sigma
    candidates &= np.isfinite(median_values)

    # 只保留长度为 1 的候选 run；真实 ramp 通常跨多个采样点，因此不会被整段压平。
    previous_is_candidate = np.r_[False, candidates[:-1]]
    next_is_candidate = np.r_[candidates[1:], False]
    isolated = candidates & ~previous_is_candidate & ~next_is_candidate
    clean = values.copy()
    clean[isolated] = median_values[isolated]
    return clean


## 2. 多尺度平台—过渡—平台检测与 J(t)

检测器不使用单点 `np.diff()` 决策。每个尺度显式比较前平台、中间过渡区和后平台，
以局部平台 MAD 标准化幅度，再检查平台持续性和过渡区的有向 Spearman 单调性。
同一物理边沿在多个尺度产生的同方向候选会合并；最终 J 在事件区间内按 ramp 变化，
事件结束后保留累计平台偏置。

In [ ]:
def _rolling_window_level_sigma(values, window):
    """一次性计算所有完整滑窗的中位数和 MAD，避免逐点重复切片。"""
    windows = np.lib.stride_tricks.sliding_window_view(values, int(window))
    levels = np.median(windows, axis=1)
    sigmas = 1.4826 * np.median(np.abs(windows - levels[:, None]), axis=1)
    return levels, sigmas


def _platform_is_persistent(pre, post, pre_level, delta, transition_samples, platform_slope_ratio):
    """同时验证平台持续性和平台自身平坦度，避免把长缓坡切成多个 J。"""
    direction = 1.0 if delta > 0 else -1.0
    midpoint = pre_level + 0.5 * delta
    pre_parts = [part for part in np.array_split(pre, 2) if part.size]
    post_parts = [part for part in np.array_split(post, 2) if part.size]
    pre_levels = [float(np.median(part)) for part in pre_parts]
    post_levels = [float(np.median(part)) for part in post_parts]
    pre_ok = all(direction * (level - midpoint) < 0 for level in pre_levels)
    post_ok = all(direction * (level - midpoint) > 0 for level in post_levels)

    # 真正的“平台—过渡—平台”要求两侧自身漂移显著小于中间跃迁幅度。
    pre_drift = abs(pre_levels[-1] - pre_levels[0])
    post_drift = abs(post_levels[-1] - post_levels[0])
    pre_rate = pre_drift / max(len(pre) / 2.0, 1.0)
    post_rate = post_drift / max(len(post) / 2.0, 1.0)
    transition_rate = abs(delta) / max(float(transition_samples), 1.0)
    flat_enough = max(pre_rate, post_rate) <= float(platform_slope_ratio) * transition_rate
    return bool(pre_ok and post_ok and flat_enough)


def detect_multiscale_transition(
    signal,
    *,
    fs,
    scales_sec,
    pre_window_sec,
    post_window_sec,
    z_threshold,
    min_duration_sec,
    monotonicity_threshold,
    platform_slope_ratio,
):
    """检测多尺度有限时长跃迁，返回事件列表和逐采样点带符号 edge score。

    confidence 是便于候选排序的 0–1 启发式分数，不代表统计概率。
    """
    values = _require_finite_signal(signal)
    if not (0 <= monotonicity_threshold <= 1):
        raise ValueError("monotonicity_threshold 必须位于 [0, 1]。")
    if z_threshold <= 0:
        raise ValueError("z_threshold 必须为正数。")
    if not (0 <= platform_slope_ratio <= 1):
        raise ValueError("platform_slope_ratio 必须位于 [0, 1]。")

    pre_n = _seconds_to_samples(pre_window_sec, fs, minimum=2)
    post_n = _seconds_to_samples(post_window_sec, fs, minimum=2)
    min_duration_n = _seconds_to_samples(min_duration_sec, fs, minimum=2)
    scale_samples = sorted(
        set(_seconds_to_samples(item, fs, minimum=2) for item in scales_sec)
    )
    if len(values) < pre_n + min(scale_samples) + post_n:
        return [], np.zeros_like(values)

    pre_levels, pre_sigmas = _rolling_window_level_sigma(values, pre_n)
    post_levels, post_sigmas = _rolling_window_level_sigma(values, post_n)

    # 全局回退尺度仅用于局部 MAD 为 0 的完全平坦窗口，不参与直接判定阶跃。
    short_window = min(len(values) if len(values) % 2 else len(values) - 1, 5)
    short_window = max(1, short_window)
    short_reference = pd.Series(values).rolling(
        short_window, center=True, min_periods=1
    ).median().to_numpy(dtype=float)
    fallback_sigma = _robust_sigma(values - short_reference)

    edge_score = np.zeros_like(values)
    candidates = []
    for transition_n in scale_samples:
        last_start = len(values) - transition_n - post_n
        if last_start < pre_n:
            continue
        starts = np.arange(pre_n, last_start + 1, dtype=int)
        pre_level = pre_levels[starts - pre_n]
        post_level = post_levels[starts + transition_n]
        delta = post_level - pre_level
        local_sigma = np.maximum(
            pre_sigmas[starts - pre_n], post_sigmas[starts + transition_n]
        )
        local_sigma = np.where(local_sigma > 0, local_sigma, fallback_sigma)
        scores = delta / np.maximum(local_sigma, np.finfo(float).eps)

        centers = starts + transition_n // 2
        better = np.abs(scores) > np.abs(edge_score[centers])
        edge_score[centers[better]] = scores[better]

        for position in np.flatnonzero(np.abs(scores) >= float(z_threshold)):
            start = int(starts[position])
            pre = values[start - pre_n : start]
            transition = values[start : start + transition_n + 1]
            post = values[start + transition_n : start + transition_n + post_n]
            event_delta = float(delta[position])
            event_pre_level = float(pre_level[position])
            if not _platform_is_persistent(
                pre, post, event_pre_level, event_delta, transition_n, platform_slope_ratio
            ):
                continue

            correlation = spearmanr(np.arange(transition.size), transition).statistic
            if not np.isfinite(correlation):
                continue
            direction_sign = 1.0 if event_delta > 0 else -1.0
            monotonicity = float(direction_sign * correlation)
            if monotonicity < float(monotonicity_threshold):
                continue

            # 以平台差的 10%/90% 首次穿越点细化事件边界，而不是直接照搬尺度宽度。
            signed_progress = direction_sign * (transition - event_pre_level)
            amplitude = abs(event_delta)
            low_hits = np.flatnonzero(signed_progress >= 0.1 * amplitude)
            high_hits = np.flatnonzero(signed_progress >= 0.9 * amplitude)
            if not low_hits.size or not high_hits.size:
                continue
            low_index = int(low_hits[0])
            high_after_low = high_hits[high_hits >= low_index]
            if not high_after_low.size:
                continue
            high_index = int(high_after_low[0])
            start_idx = start + low_index
            end_idx = start + high_index
            if end_idx - start_idx < min_duration_n:
                continue

            score_value = float(scores[position])
            z_confidence = 1.0 - math.exp(
                -max(abs(score_value) - float(z_threshold), 0.0) / float(z_threshold)
            )
            confidence = float(np.clip(0.5 * z_confidence + 0.5 * monotonicity, 0, 1))
            candidates.append(
                {
                    "start": start_idx / float(fs),
                    "end": end_idx / float(fs),
                    "duration": (end_idx - start_idx) / float(fs),
                    "delta": event_delta,
                    "direction": "rise" if event_delta > 0 else "fall",
                    "confidence": confidence,
                    "start_idx": start_idx,
                    "end_idx": end_idx,
                    "scale_sec": transition_n / float(fs),
                    "edge_score": score_value,
                    "monotonicity": monotonicity,
                }
            )

    # 非极大值抑制：同方向且中心足够接近的多尺度候选视为同一物理事件。
    selected = []
    for candidate in sorted(
        candidates, key=lambda item: (item["confidence"], abs(item["edge_score"])), reverse=True
    ):
        center = 0.5 * (candidate["start"] + candidate["end"])
        duplicate = False
        for accepted in selected:
            accepted_center = 0.5 * (accepted["start"] + accepted["end"])
            same_direction = candidate["direction"] == accepted["direction"]
            merge_distance = max(candidate["scale_sec"], accepted["scale_sec"])
            if same_direction and abs(center - accepted_center) <= merge_distance:
                duplicate = True
                break
        if not duplicate:
            selected.append(candidate)

    events = sorted(selected, key=lambda item: (item["start_idx"], item["end_idx"]))
    for event in events:
        lo, hi = event["start_idx"], event["end_idx"] + 1
        segment = edge_score[lo:hi]
        replace = np.abs(segment) < abs(event["edge_score"])
        segment[replace] = event["edge_score"]
        edge_score[lo:hi] = segment
    return events, edge_score


def build_transition_component(length, *, fs, events, mask_margin_sec):
    """把事件重建为可相加的累计 ramp J，并生成供趋势估计使用的布尔 mask。"""
    if int(length) <= 0:
        raise ValueError("length 必须为正整数。")
    margin = max(0, int(round(float(mask_margin_sec) * int(fs))))
    component = np.zeros(int(length), dtype=float)
    mask = np.zeros(int(length), dtype=bool)

    # 每个事件独立生成“0 → delta → 持续保持”的贡献，再线性相加；因此反向事件会自然抵消。
    for event in sorted(events, key=lambda item: item["start_idx"]):
        start = int(np.clip(event["start_idx"], 0, length - 1))
        end = int(np.clip(event["end_idx"], start, length - 1))
        delta = float(event["delta"])
        component[start : end + 1] += np.linspace(0.0, delta, end - start + 1)
        if end + 1 < length:
            component[end + 1 :] += delta
        mask[max(0, start - margin) : min(length, end + margin + 1)] = True
    return component, mask


### 阶段二关键契约

固定随机种子的合成信号用于防止三个关键回归：误删真实 ramp、把 ramp 当单点阶跃、以及 J 在事件结束后没有保留平台偏置。

In [ ]:
rng = np.random.default_rng(7)
synthetic_fs = 100
synthetic_time = np.arange(800) / synthetic_fs
synthetic_raw = rng.normal(0.0, 0.002, synthetic_time.size)
ramp_start, ramp_end = 300, 320
synthetic_raw[ramp_start : ramp_end + 1] += np.linspace(0.0, 0.5, ramp_end - ramp_start + 1)
synthetic_raw[ramp_end + 1 :] += 0.5
synthetic_raw[100] += 0.15  # 明显的单采样点尖峰

synthetic_clean = hampel_filter(
    synthetic_raw, fs=synthetic_fs, window_sec=HAMPEL_WINDOW_SEC, n_sigma=HAMPEL_N_SIGMA
)
assert abs(synthetic_clean[100]) < abs(synthetic_raw[100]) * 0.25, "孤立尖峰没有被充分替换"

synthetic_events, synthetic_edge_score = detect_multiscale_transition(
    synthetic_clean,
    fs=synthetic_fs,
    scales_sec=EDGE_SCALES_SEC,
    pre_window_sec=EDGE_PRE_WINDOW_SEC,
    post_window_sec=EDGE_POST_WINDOW_SEC,
    z_threshold=EDGE_Z_THRESHOLD,
    min_duration_sec=EDGE_MIN_DURATION_SEC,
    monotonicity_threshold=EDGE_MONOTONICITY_THRESHOLD,
    platform_slope_ratio=EDGE_PLATFORM_SLOPE_RATIO,
)
rise_events = [event for event in synthetic_events if event["direction"] == "rise"]
assert rise_events, "未检测到合成 ramp"
synthetic_event = max(rise_events, key=lambda event: event["confidence"])
assert synthetic_event["duration"] >= EDGE_MIN_DURATION_SEC, "事件被错误压缩为单点阶跃"
assert abs(synthetic_event["delta"] - 0.5) < 0.05, "平台差估计偏离合成真值"

synthetic_J, synthetic_J_mask = build_transition_component(
    len(synthetic_clean), fs=synthetic_fs, events=[synthetic_event],
    mask_margin_sec=EDGE_MASK_MARGIN_SEC,
)
synthetic_dejumped = synthetic_clean - synthetic_J
original_gap = abs(np.median(synthetic_clean[340:390]) - np.median(synthetic_clean[240:290]))
remaining_gap = abs(np.median(synthetic_dejumped[340:390]) - np.median(synthetic_dejumped[240:290]))
assert remaining_gap < 0.15 * original_gap, "减去 J 后平台仍不连续"
assert abs(np.median(synthetic_J[400:450]) - synthetic_event["delta"]) < 0.02

print(
    f"阶段二合成检查通过：duration={synthetic_event['duration']:.3f}s, "
    f"delta={synthetic_event['delta']:.3f}, 平台残差比例={remaining_gap/original_gap:.3f}"
)

### 在当前真实样本上检查 J

两路使用相同参数和函数，但 Hampel、局部 MAD、候选及事件全部独立计算。

In [ ]:
def _run_transition_stage(signal):
    """以首个参数单元中的统一配置运行轻度清理和 J 检测。"""
    clean = hampel_filter(
        signal, fs=fs, window_sec=HAMPEL_WINDOW_SEC, n_sigma=HAMPEL_N_SIGMA
    )
    events, score = detect_multiscale_transition(
        clean,
        fs=fs,
        scales_sec=EDGE_SCALES_SEC,
        pre_window_sec=EDGE_PRE_WINDOW_SEC,
        post_window_sec=EDGE_POST_WINDOW_SEC,
        z_threshold=EDGE_Z_THRESHOLD,
        min_duration_sec=EDGE_MIN_DURATION_SEC,
        monotonicity_threshold=EDGE_MONOTONICITY_THRESHOLD,
    platform_slope_ratio=EDGE_PLATFORM_SLOPE_RATIO,
    )
    J, mask = build_transition_component(
        len(clean), fs=fs, events=events, mask_margin_sec=EDGE_MASK_MARGIN_SEC
    )
    return clean, events, score, J, mask, clean - J


clean2, events2, edge_score2, J2, J_mask2, dejumped2 = _run_transition_stage(udeco2)
clean3, events3, edge_score3, J3, J_mask3, dejumped3 = _run_transition_stage(udeco3)

required_event_fields = {"start", "end", "duration", "delta", "direction", "confidence"}
for channel_name, events in (("Udeco2", events2), ("Udeco3", events3)):
    assert all(required_event_fields <= event.keys() for event in events)
    print(f"{channel_name}: 检测到 {len(events)} 个 J 事件")
    display(pd.DataFrame(events).head(10))

## 3. J 校正后的 Robust LOWESS 中尺度趋势

先计算 `dejumped = clean - J`，再把 `J_mask` 内的短缺口用两侧数据线性连接，
得到只供 LOWESS 使用的 `medium_input`。这样做不是修改真实信号，而是避免 J 的估计误差
在局部回归中再次形成边沿。LOWESS 的窗口由秒换算成记录长度占比，稳健迭代降低残余离群点影响。

In [ ]:
def estimate_medium_trend(
    signal,
    j_mask,
    *,
    fs,
    method,
    window_sec,
    robust_iters,
):
    """在屏蔽快速事件后估计 Bm，并返回插值代理输入、Bm 和 dBm/dt。"""
    values = _require_finite_signal(signal, name="dejumped")
    mask = np.asarray(j_mask, dtype=bool).reshape(-1)
    if mask.shape != values.shape:
        raise ValueError("j_mask 必须与 signal 等长。")
    if method != "robust_loess":
        raise ValueError("第一版仅支持 MEDIUM_METHOD='robust_loess'。")
    if int(robust_iters) < 0:
        raise ValueError("robust_iters 不能为负数。")

    sample_index = np.arange(values.size)
    valid = ~mask
    if np.count_nonzero(valid) < 2:
        raise ValueError("J_mask 屏蔽了几乎全部数据，无法估计中尺度趋势。")
    medium_input = values.copy()
    # np.interp 在记录边缘使用最近的有效值，在内部短 mask 上执行线性连接。
    medium_input[mask] = np.interp(sample_index[mask], sample_index[valid], values[valid])

    window_samples = _seconds_to_samples(window_sec, fs, minimum=3)
    minimum_fraction = min(1.0, 3.0 / values.size)
    fraction = float(np.clip(window_samples / values.size, minimum_fraction, 1.0))
    x = sample_index / float(fs)
    Bm = np.asarray(
        lowess(
            endog=medium_input,
            exog=x,
            frac=fraction,
            it=int(robust_iters),
            delta=0.0,
            is_sorted=True,
            missing="raise",
            return_sorted=False,
        ),
        dtype=float,
    )
    if Bm.shape != values.shape or not np.isfinite(Bm).all():
        raise RuntimeError("Robust LOWESS 未返回完整的有限 Bm。")
    dBm_dt = np.gradient(Bm, 1.0 / float(fs))
    return medium_input, Bm, dBm_dt


## 4. 条件冻结的超慢 Bslow 与连续压迫量 P

超慢基线不是巨大窗口低通。它只在 `J_mask=False`、边沿分数低、Bm 斜率低且 Bm
仍接近当前长期参考时缓慢更新；其他时刻保持上一值。由此，压迫缓坡和高平台保留在
$P(t)=B_m(t)-B_{\mathrm{slow}}(t)$ 中，而不会随着时间被无条件吸入基线。

In [ ]:
def estimate_conditional_bslow(
    Bm,
    edge_score,
    j_mask,
    dBm_dt,
    *,
    fs,
    tau_sec,
    update_rate,
    stable_edge_threshold,
    stable_slope_threshold,
    stable_offset_threshold,
):
    """逐点执行条件 EWMA，返回 Bslow 和实际发生更新的位置。"""
    trend = _require_finite_signal(Bm, name="Bm")
    edge = _require_finite_signal(edge_score, name="edge_score")
    slope = _require_finite_signal(dBm_dt, name="dBm_dt")
    mask = np.asarray(j_mask, dtype=bool).reshape(-1)
    if not (trend.shape == edge.shape == slope.shape == mask.shape):
        raise ValueError("Bm、edge_score、J_mask 和 dBm_dt 必须等长。")
    if tau_sec <= 0 or not (0 < update_rate <= 1):
        raise ValueError("tau_sec 必须为正数，update_rate 必须位于 (0, 1]。")
    if min(stable_edge_threshold, stable_slope_threshold, stable_offset_threshold) < 0:
        raise ValueError("Bslow 的三个稳定阈值不能为负数。")

    alpha = float(update_rate) * (1.0 - math.exp(-1.0 / (float(fs) * float(tau_sec))))
    Bslow = np.empty_like(trend)
    update_mask = np.zeros_like(mask)
    Bslow[0] = trend[0]

    for index in range(1, trend.size):
        previous = Bslow[index - 1]
        stable = (
            (not mask[index])
            and abs(edge[index]) < stable_edge_threshold
            and abs(slope[index]) < stable_slope_threshold
            and abs(trend[index] - previous) < stable_offset_threshold
        )
        if stable:
            Bslow[index] = previous + alpha * (trend[index] - previous)
            update_mask[index] = True
        else:
            # 不稳定时使用精确赋值而不是很小的 alpha，便于确认基线确实冻结。
            Bslow[index] = previous
    return Bslow, update_mask


def decompose_udeco(
    signal,
    fs,
    *,
    hampel_window_sec,
    hampel_n_sigma,
    edge_scales_sec,
    edge_pre_window_sec,
    edge_post_window_sec,
    edge_z_threshold,
    edge_min_duration_sec,
    edge_monotonicity_threshold,
    edge_platform_slope_ratio,
    edge_mask_margin_sec,
    medium_method,
    medium_window_sec,
    medium_robust_iters,
    bslow_tau_sec,
    bslow_update_rate,
    bslow_stable_edge_threshold,
    bslow_stable_slope_threshold,
    bslow_stable_offset_threshold,
):
    """以同一接口完成单路 Udeco 的 J、P、Bslow 分解。"""
    raw = _require_finite_signal(signal).copy()
    clean = hampel_filter(
        raw, fs=fs, window_sec=hampel_window_sec, n_sigma=hampel_n_sigma
    )
    events, edge_score = detect_multiscale_transition(
        clean,
        fs=fs,
        scales_sec=edge_scales_sec,
        pre_window_sec=edge_pre_window_sec,
        post_window_sec=edge_post_window_sec,
        z_threshold=edge_z_threshold,
        min_duration_sec=edge_min_duration_sec,
        monotonicity_threshold=edge_monotonicity_threshold,
        platform_slope_ratio=edge_platform_slope_ratio,
    )
    J, J_mask = build_transition_component(
        len(clean), fs=fs, events=events, mask_margin_sec=edge_mask_margin_sec
    )
    dejumped = clean - J
    medium_input, Bm, dBm_dt = estimate_medium_trend(
        dejumped,
        J_mask,
        fs=fs,
        method=medium_method,
        window_sec=medium_window_sec,
        robust_iters=medium_robust_iters,
    )
    Bslow, bslow_update_mask = estimate_conditional_bslow(
        Bm,
        edge_score,
        J_mask,
        dBm_dt,
        fs=fs,
        tau_sec=bslow_tau_sec,
        update_rate=bslow_update_rate,
        stable_edge_threshold=bslow_stable_edge_threshold,
        stable_slope_threshold=bslow_stable_slope_threshold,
        stable_offset_threshold=bslow_stable_offset_threshold,
    )
    P = Bm - Bslow
    residual = clean - J - P - Bslow
    return {
        "raw": raw,
        "clean": clean,
        "J": J,
        "J_events": events,
        "J_mask": J_mask,
        "edge_score": edge_score,
        "dejumped": dejumped,
        "medium_input": medium_input,
        "Bm": Bm,
        "dBm_dt": dBm_dt,
        "Bslow": Bslow,
        "bslow_update_mask": bslow_update_mask,
        "P": P,
        "residual": residual,
    }


### 阶段三关键合成检查

这个检查只覆盖最重要的分层契约：J 相减后的连续性、Bm 对压迫缓坡的保留、Bslow 在压迫/J 区域冻结，以及加性重建恒等式。

In [ ]:
decomposition_kwargs = {
    "hampel_window_sec": HAMPEL_WINDOW_SEC,
    "hampel_n_sigma": HAMPEL_N_SIGMA,
    "edge_scales_sec": EDGE_SCALES_SEC,
    "edge_pre_window_sec": EDGE_PRE_WINDOW_SEC,
    "edge_post_window_sec": EDGE_POST_WINDOW_SEC,
    "edge_z_threshold": EDGE_Z_THRESHOLD,
    "edge_min_duration_sec": EDGE_MIN_DURATION_SEC,
    "edge_monotonicity_threshold": EDGE_MONOTONICITY_THRESHOLD,
    "edge_platform_slope_ratio": EDGE_PLATFORM_SLOPE_RATIO,
    "edge_mask_margin_sec": EDGE_MASK_MARGIN_SEC,
    "medium_method": MEDIUM_METHOD,
    "medium_window_sec": MEDIUM_WINDOW_SEC,
    "medium_robust_iters": MEDIUM_ROBUST_ITERS,
    "bslow_tau_sec": BSLOW_TAU_SEC,
    "bslow_update_rate": BSLOW_UPDATE_RATE,
    "bslow_stable_edge_threshold": BSLOW_STABLE_EDGE_THRESHOLD,
    "bslow_stable_slope_threshold": BSLOW_STABLE_SLOPE_THRESHOLD,
    "bslow_stable_offset_threshold": BSLOW_STABLE_OFFSET_THRESHOLD,
}

rng = np.random.default_rng(17)
check_fs = 100
check_time = np.arange(6000) / check_fs
true_bslow = 1e-6 * check_time
true_P = np.zeros_like(check_time)
true_P[1000:2001] = np.linspace(0.0, 1e-3, 1001)
true_P[2001:3000] = 1e-3
true_P[3000:4001] = np.linspace(1e-3, 0.0, 1001)
true_J = np.zeros_like(check_time)
true_J[4500:4521] = np.linspace(0.0, 2e-3, 21)
true_J[4521:] = 2e-3
check_signal = true_bslow + true_P + true_J + rng.normal(0.0, 1e-5, check_time.size)

check_result = decompose_udeco(check_signal, check_fs, **decomposition_kwargs)
assert check_result["J_events"], "合成快速 ramp 未被检测"

original_gap = abs(np.median(check_result["clean"][4700:4900]) - np.median(check_result["clean"][4200:4400]))
remaining_gap = abs(np.median(check_result["dejumped"][4700:4900]) - np.median(check_result["dejumped"][4200:4400]))
assert remaining_gap < 0.2 * original_gap, "J 相减后平台仍有明显断点"

medium_slice = slice(800, 4200)
medium_truth = true_bslow + true_P
medium_correlation = float(np.corrcoef(check_result["Bm"][medium_slice], medium_truth[medium_slice])[0, 1])
estimated_amplitude = float(np.median(check_result["Bm"][2200:2800]) - np.median(check_result["Bm"][200:800]))
true_amplitude = float(np.median(medium_truth[2200:2800]) - np.median(medium_truth[200:800]))
assert medium_correlation > 0.9, "Bm 未保留中尺度压迫形态"
assert estimated_amplitude > 0.6 * true_amplitude, "Bm 对压迫缓坡削弱过多"

update_mask = check_result["bslow_update_mask"]
assert update_mask[200:800].mean() > 0.5, "长期稳定段没有正常打开 Bslow 更新"
assert update_mask[2200:2800].mean() < 0.1, "压迫高平台被错误允许更新 Bslow"
assert not update_mask[check_result["J_mask"]].any(), "J mask 内不应更新 Bslow"
frozen_steps = ~update_mask[1:]
assert np.all(np.diff(check_result["Bslow"])[frozen_steps] == 0.0), "冻结区 Bslow 仍在变化"

reconstructed = check_result["J"] + check_result["P"] + check_result["Bslow"] + check_result["residual"]
assert np.allclose(check_result["clean"], reconstructed, rtol=1e-10, atol=1e-12)
print(
    f"阶段三合成检查通过：Bm correlation={medium_correlation:.4f}, "
    f"缓坡保留={estimated_amplitude/true_amplitude:.3f}, "
    f"稳定段更新率={update_mask[200:800].mean():.3f}"
)

### 分别分解当前样本的 Udeco2 / Udeco3

两个调用共享完全相同的参数字典，但返回互不共享的数组、局部噪声估计、事件和 update mask。

In [ ]:
result2 = decompose_udeco(udeco2, fs, **decomposition_kwargs)
result3 = decompose_udeco(udeco3, fs, **decomposition_kwargs)

required_result_fields = {
    "raw", "clean", "J", "J_events", "J_mask", "edge_score",
    "dejumped", "medium_input", "Bm", "dBm_dt", "Bslow",
    "bslow_update_mask", "P", "residual",
}
summary_rows = []
for channel_name, result in (("Udeco2", result2), ("Udeco3", result3)):
    assert required_result_fields <= result.keys()
    for key in ("raw", "clean", "J", "J_mask", "edge_score", "dejumped", "medium_input", "Bm", "dBm_dt", "Bslow", "bslow_update_mask", "P", "residual"):
        assert len(result[key]) == len(time), f"{channel_name}.{key} 长度不匹配"
    numeric_keys = ("raw", "clean", "J", "edge_score", "dejumped", "medium_input", "Bm", "dBm_dt", "Bslow", "P", "residual")
    assert all(np.isfinite(result[key]).all() for key in numeric_keys)
    reconstructed = result["J"] + result["P"] + result["Bslow"] + result["residual"]
    assert np.allclose(result["clean"], reconstructed, rtol=1e-10, atol=1e-12)
    frozen_steps = ~result["bslow_update_mask"][1:]
    assert np.all(np.diff(result["Bslow"])[frozen_steps] == 0.0)
    summary_rows.append(
        {
            "channel": channel_name,
            "J_events": len(result["J_events"]),
            "J_mask_ratio": float(result["J_mask"].mean()),
            "Bslow_update_ratio": float(result["bslow_update_mask"].mean()),
            "P_min": float(result["P"].min()),
            "P_max": float(result["P"].max()),
            "residual_robust_sigma": _robust_sigma(result["residual"]),
        }
    )

display(pd.DataFrame(summary_rows))